# Core Memory Pattern — Agent-Managed Structured Memory

This demo uses Strands Agents SDK. The core memory patterns demonstrated here are framework-agnostic and can be applied with LangGraph, AutoGen, or other agent frameworks that support tool-based state management.

Based on:
- [MIRIX: Multi-Agent Memory System](https://arxiv.org/abs/2507.07957) — Wang & Chen, 2025
- [MemGPT: Towards LLMs as Operating Systems](https://arxiv.org/abs/2310.08560) — Packer et al., 2023
- [Enabling Personalized Long-term Interactions](https://arxiv.org/abs/2510.07925) — Westhaeusser et al., 2025

## The Problem

In Demo 01, tools automatically write preferences to `agent.state`. This works, but the agent has no **awareness** of its own memory. It cannot:
- Decide what information is worth remembering
- Update outdated preferences when they change
- Organize knowledge into structured sections
- Reflect on what it knows before answering

## The Solution: Core Memory Tools

Give the agent explicit tools to manage its own memory:

| Tool | Purpose |
|------|---------|
| `core_memory_read(section)` | Read a named memory section |
| `core_memory_write(section, content)` | Create a new section |
| `core_memory_update(section, updates)` | Merge updates into existing section |
| `core_memory_list()` | List all sections with metadata |

The agent determines when to store and retrieve information based on conversation context and explicit memory management tools.

## What We Test

| Test | Approach | Core memory | Evolves | Persists |
|------|----------|-------------|---------|----------|
| 1 — No memory tools | Baseline | Empty | — | — |
| 2 — Core memory tools | Read/write/update | Populated | — | — |
| 3 — Evolution | Preferences change | Updated | Yes | — |
| 4 — Persistence | FileSessionManager | Restored | Yes | Yes |

## Configure API Key

Set your OpenAI API key. Get one at https://platform.openai.com/api-keys

> You can swap to any supported model provider. Change the model in the setup cell below.

In [ ]:
import os
# os.environ['OPENAI_API_KEY'] = 'your-key-here'  # Uncomment and set your key
assert os.getenv('OPENAI_API_KEY'), (
    'OPENAI_API_KEY not set. '
    'Get yours at https://platform.openai.com/api-keys and set it above or in a .env file.'
)

## Setup

In [ ]:
import json, time, os, shutil

os.environ['OTEL_SDK_DISABLED'] = 'true'

from dotenv import load_dotenv
from strands import Agent, tool, ToolContext
# Using OpenAI-compatible interface via Strands SDK (not direct OpenAI usage)
from strands.models.openai import OpenAIModel
from strands.agent.conversation_manager import SlidingWindowConversationManager
from strands.session import FileSessionManager
from tools import (
    core_memory_read, core_memory_write, core_memory_update, core_memory_list,
    search_hotels_with_memory, book_hotel_with_memory,
)

load_dotenv()

MODEL = OpenAIModel(model_id='gpt-4o-mini')

SYSTEM_PROMPT_NO_MEMORY = (
    'You are a travel assistant. Help users find and book hotels. '
    'Be concise — answer in 2-3 sentences maximum.'
)

SYSTEM_PROMPT_CORE_MEMORY = (
    'You are a travel assistant with core memory capabilities. '
    'You can read and write to your core memory to remember user preferences. '
    '\n\nIMPORTANT memory management rules:'
    '\n1. When a user shares personal info (name, preferences), WRITE it to core_memory persona section'
    '\n2. When a user books something, UPDATE the preferences section with learned preferences'
    '\n3. Before searching, READ preferences to personalize results'
    '\n4. Use core_memory_list to check what you already know'
    '\n\nBe concise — answer in 2-3 sentences maximum.'
)

MEMORY_TOOLS = [
    core_memory_read, core_memory_write, core_memory_update, core_memory_list,
    search_hotels_with_memory, book_hotel_with_memory,
]

print('Setup complete!')

---
## Test 1 — No Core Memory (Baseline)

The agent has hotel tools but NO core memory tools. When the user shares preferences, the agent has no mechanism to store them structurally.

**Expected:** `core_memory` in `agent.state` remains empty. The agent cannot recall preferences on the next turn.

In [ ]:
agent_no_memory = Agent(
    model=MODEL,
    system_prompt=SYSTEM_PROMPT_NO_MEMORY,
    tools=[search_hotels_with_memory, book_hotel_with_memory],
)

print('Turn 1: My name is Alex. I love traditional Japanese culture and prefer 4-star hotels.')
agent_no_memory('My name is Alex. I love traditional Japanese culture and prefer 4-star hotels.')

print('\nTurn 2: Search hotels in Tokyo for me')
agent_no_memory('Search hotels in Tokyo for me')

memory = agent_no_memory.state.get('core_memory')
print(f'\ncore_memory: {json.dumps(memory) if memory else "Empty — no memory tools available"}')

---
## Test 2 — With Core Memory Tools

Now the agent has `core_memory_write`, `core_memory_read`, `core_memory_update`, and `core_memory_list`. The system prompt instructs it to:

1. **WRITE** persona info when the user shares it
2. **UPDATE** preferences when the user books something
3. **READ** preferences before searching to personalize results

```python
@tool(context=True)
def core_memory_write(section: str, content: str, tool_context: ToolContext) -> str:
    memory = tool_context.agent.state.get('core_memory') or {}
    memory[section] = {'data': json.loads(content), 'version': 1, ...}
    tool_context.agent.state.set('core_memory', memory)
```

**Expected:** Agent stores persona in core memory, then uses it to rank Tokyo hotels. After booking, it updates preferences.

In [ ]:
agent_core = Agent(
    model=MODEL,
    system_prompt=SYSTEM_PROMPT_CORE_MEMORY,
    conversation_manager=SlidingWindowConversationManager(window_size=40),
    tools=MEMORY_TOOLS,
)

print('Turn 1: My name is Alex. I love traditional Japanese culture and prefer 4-star hotels.')
agent_core('My name is Alex. I love traditional Japanese culture and prefer 4-star hotels.')

print('\nTurn 2: Search hotels in Tokyo for me')
agent_core('Search hotels in Tokyo for me')

print('\nTurn 3: Book the Zen Garden Ryokan for 3 nights')
agent_core('Book the Zen Garden Ryokan for 3 nights')

print('\nTurn 4: Now search Zurich hotels — what fits my profile?')
agent_core('Now search Zurich hotels — what fits my profile?')

# Show what core memory looks like
memory = agent_core.state.get('core_memory') or {}
print(f'\nCore memory ({len(memory)} sections):')
for section, data in memory.items():
    print(f'  [{section}] v{data.get("version", "?")}: {json.dumps(data.get("data", ""))[:100]}...')

---
## Test 3 — Core Memory Evolution

User preferences change over time. Core memory must evolve — not just append.

`core_memory_update` **merges** new data with existing data and increments the version counter. The agent tracks that preferences changed from luxury to boutique.

**Expected:** After Turn 2, the preferences section reflects the updated style (boutique, not luxury). Version number increments.

In [ ]:
agent_evolve = Agent(
    model=MODEL,
    system_prompt=SYSTEM_PROMPT_CORE_MEMORY,
    conversation_manager=SlidingWindowConversationManager(window_size=40),
    tools=MEMORY_TOOLS,
)

print('Turn 1: I\'m Alex, I prefer luxury 5-star hotels with spa access')
agent_evolve("I'm Alex, I prefer luxury 5-star hotels with spa access")

v1 = json.dumps(agent_evolve.state.get('core_memory') or {})
print(f'\nMemory after Turn 1: {v1[:150]}...')

print('\nTurn 2: Actually, I now prefer boutique hotels under $250 with garden access')
agent_evolve('Actually, I now prefer boutique hotels under $250 with character and garden access')

v2 = json.dumps(agent_evolve.state.get('core_memory') or {})
print(f'\nMemory after Turn 2: {v2[:150]}...')
print(f'Memory evolved: {v1 != v2}')

print('\nTurn 3: What are my current preferences?')
agent_evolve('What are my current preferences? Read my profile from core memory.')

---
## Test 4 — Cross-Session Persistence

Core memory built in Session A is persisted via `FileSessionManager`. When Session B starts (new agent instance, same `session_id`), the memory is fully restored.

```python
agent = Agent(
    model=MODEL,
    tools=MEMORY_TOOLS,
    session_manager=FileSessionManager(session_id='user-42', storage_dir='./sessions'),
)
# core_memory is automatically restored from disk
```

**Expected:** Session B agent knows the user's name, preferences, and booking history without being told again.

In [ ]:
session_id = 'core-memory-notebook-demo'
storage_dir = os.path.join(os.path.dirname(os.path.abspath('.')), 'sessions')

# --- Session A ---
print('--- Session A: Build core memory ---')
agent_a = Agent(
    model=MODEL,
    system_prompt=SYSTEM_PROMPT_CORE_MEMORY,
    conversation_manager=SlidingWindowConversationManager(window_size=40),
    tools=MEMORY_TOOLS,
    session_manager=FileSessionManager(session_id=session_id, storage_dir=storage_dir),
)

agent_a("I'm Alex, I love traditional Japanese culture, prefer 4-star hotels with onsen and garden")
agent_a('Book the Zen Garden Ryokan for 3 nights')

memory_a = agent_a.state.get('core_memory')
print(f'\nSession A sections: {list(memory_a.keys()) if memory_a else "none"}')

# --- Session B ---
print('\n--- Session B: Returning user (new agent instance) ---')
agent_b = Agent(
    model=MODEL,
    system_prompt=SYSTEM_PROMPT_CORE_MEMORY,
    conversation_manager=SlidingWindowConversationManager(window_size=40),
    tools=MEMORY_TOOLS,
    session_manager=FileSessionManager(session_id=session_id, storage_dir=storage_dir),
)

memory_b = agent_b.state.get('core_memory')
print(f'Session B sections restored: {list(memory_b.keys()) if memory_b else "none"}')
print(f'State survived restart: {memory_a == memory_b}')

print('\nAsking agent: What do you remember about me?')
agent_b('What do you remember about me? Check your core memory.')

# Cleanup
if os.path.exists(storage_dir):
    shutil.rmtree(storage_dir)
    print('\n(Session files cleaned up)')

---
## Summary

| Test | Core memory | Evolves | Persists |
|------|-------------|---------|----------|
| 1 — No memory tools | Empty | — | — |
| 2 — Core memory tools | Populated | — | — |
| 3 — Evolution | Updated | Yes | — |
| 4 — Persistence | Restored | Yes | Yes |

## Key Takeaways

- Core memory gives the agent **agency** over its own memory
- The system prompt drives memory management behavior
- `core_memory_update` merges data (not overwrites) and tracks versions
- `FileSessionManager` persists core memory across sessions automatically

## Next Steps

1. [Demo 01: Memory Decay](../01-memory-decay-demo/) — Why agents forget (the problem)
2. [Demo 03: Memory Retrieval](../03-memory-retrieval-demo/) — When memory grows large, find the right memories

## References

### Research
- [MIRIX: Multi-Agent Memory System](https://arxiv.org/abs/2507.07957) — 6 memory types, SOTA 85.4% on LOCOMO
- [MemGPT: Towards LLMs as Operating Systems](https://arxiv.org/abs/2310.08560) — Core memory concept
- [Enabling Personalized Long-term Interactions](https://arxiv.org/abs/2510.07925) — Persistent memory + user profiles

### Implementation Resources
- [Agent State API](https://github.com/strands-agents/sdk-python#agent-state) — State management used in this demo
- [Session Management](https://github.com/strands-agents/sdk-python#sessions) — Cross-session persistence
- [Code Repository](https://github.com/aws-samples/sample-why-agents-fail)